# SPK-6 — Adaptive Query Execution (AQE) deep-dive

**Break → Detect → Fix → Prove.** AQE re-plans a query *at runtime* using the real statistics each
shuffle produces, instead of trusting the optimizer's compile-time guesses. We run **three demos**,
each **AQE-off vs AQE-on**, reading the effect from `df.explain()` and a before/after metrics table.

This is the deep-dive companion to the skew flagship [`SPK-1`](../skew/README.md): SPK-1 fixes skew
by hand and uses AQE skew-join as *one* of three remedies — here we open AQE up and watch **all** of
what it does for free (coalesce, skew-split, SMJ→broadcast re-optimize), plus the two places it costs you.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch the **SQL / DataFrame** tab as the
cells run.

**Laptop-safe:** data is *generated lazily* (10–20M rows) and only `count()`-ed — never collected or
written — so nothing fills memory or disk. AQE behavior is about task/partition counts and plan
shape, not memory, so the default **tuned** box is fine (no need for `make up-constrained`). Nothing
to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [1]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import skewed_keys, key_dimension, uniform_keys
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run — AQE shows up in the SQL / DataFrame tab's final plan.
print("Spark UI:", "http://localhost:4040")
spark

Spark UI: http://localhost:4040


## Step 0 — A tiny helper to read AQE off the plan

We stay **Connect-safe**: DataFrame/SQL, `spark.conf`, and `df.explain()` only — never
`spark.sparkContext` / RDDs. The AQE tells live in the *final physical plan*:

- **`AdaptiveSparkPlan isFinalPlan=true`** — AQE ran and settled on a final plan.
- **`AQEShuffleRead`** — a coalesced (fewer partitions) or skew-split (more partitions) shuffle read.
- the **join operator** (`SortMergeJoin` vs `BroadcastHashJoin`) — what AQE chose at runtime.

`explain()` only shows `isFinalPlan=true` *after* the query has executed, so we call it on a frame
we've already `count()`-ed.

In [2]:
def scan_plan(df, note=""):
    """Print df's physical plan and flag the AQE markers (Connect-safe: explain() only)."""
    plan = df._explain_string(mode="formatted") if hasattr(df, "_explain_string") else None
    if plan is None:
        # Fallback that works on all Spark Connect builds: capture explain() output.
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            df.explain(mode="formatted")
        plan = buf.getvalue()
    print(f"--- physical plan {note} ---")
    print(plan)
    print("AQE markers:",
          "isFinalPlan=true"     if "isFinalPlan=true" in plan else "(not final yet)",
          "| AQEShuffleRead"      if "AQEShuffleRead"   in plan else "| no AQEShuffleRead",
          "| BroadcastHashJoin"   if "BroadcastHashJoin" in plan else
          "| SortMergeJoin"       if "SortMergeJoin"     in plan else "")
    return plan

## Demo (a) — Coalesce shuffle partitions

A heavily-*filtered* fact aggregated with `spark.sql.shuffle.partitions = 200`. The shuffle produces
**200 mostly-empty partitions → 200 tiny tasks** for a few rows of result. AQE's
`coalescePartitions` collapses them to a handful sized to the *real* post-filter data.

**Break it (AQE off):** the `constrained` profile sets `adaptive.enabled=false` and
`coalescePartitions.enabled=false`. We pin 200 partitions and aggregate a tiny slice of a 20M-row fact.

In [3]:
N_ROWS = 20_000_000

# A uniform fact; filter to a tiny slice, then aggregate -> a small post-shuffle result.
fact_a = uniform_keys(spark, n_rows=N_ROWS, n_keys=2_000, key_col="customer_id")
agg_a  = (fact_a.filter(F.col("customer_id") < 20)        # throw away ~99% of rows
                .groupBy("customer_id")
                .agg(F.sum("amount").alias("total")))

apply_profile(spark, "constrained", **{"spark.sql.shuffle.partitions": "200"})  # 200 tiny tasks
m_a_off = measure(spark, "coalesce: AQE off (200)", lambda: agg_a.count())
scan_plan(agg_a, "(a) AQE OFF")
print("\nmetrics:", m_a_off)

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 200
--- physical plan (a) AQE OFF ---
== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Filter (3)
         +- * Project (2)
            +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#185L]
Arguments: Range (0, 20000000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [2]: [pmod(id#185L, 2000) AS customer_id#188L, (rand(17) * 100.0) AS amount#190]
Input [1]: [id#185L]

(3) Filter [codegen id : 1]
Input [2]: [customer_id#188L, amount#190]
Condition : (isnotnull(customer_id#188L) AND (customer_id#188L < 20))

(4) HashAggregate [codegen id : 1]
Input [2]: [customer_id#188L, amount#190]
Keys [1]: [custo

**Detect** (Spark UI → SQL/DataFrame): AQE-off the `Exchange` feeds ~200 tasks (Stages tab shows
hundreds of millisecond tasks). **Fix it (AQE on):** `tuned` turns on `coalescePartitions`. The plan
gains an **`AQEShuffleRead coalesced`** node and the task count drops to single digits — even though
we *still* request 200 partitions, AQE right-sizes them at runtime.

In [4]:
apply_profile(spark, "tuned", **{"spark.sql.shuffle.partitions": "200"})  # AQE will coalesce these
m_a_on = measure(spark, "coalesce: AQE on", lambda: agg_a.count())
scan_plan(agg_a, "(a) AQE ON")
print("\nmetrics:", m_a_on)
print("\n>>> Prove (a): task count should fall sharply with AQE on")
compare([m_a_off, m_a_on])

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
--- physical plan (a) AQE ON ---
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Filter (3)
            +- Project (2)
               +- Range (1)


(1) Range
Output [1]: [id#185L]
Arguments: Range (0, 20000000, step=1, splits=Some(8))

(2) Project
Output [2]: [pmod(id#185L, 2000) AS customer_id#188L, (rand(17) * 100.0) AS amount#190]
Input [1]: [id#185L]

(3) Filter
Input [2]: [customer_id#188L, amount#190]
Condition : (isnotnull(customer_id#188L) AND (customer_id#188L < 20))

(4) HashAggregate
Input [2]: [customer_id#188L, amount#190]
Keys [1]: [customer_id#188L]
Functions [1]: [partial_sum(a

## Demo (b) — Skew-join split  *(cross-ref [`SPK-1`](../skew/README.md))*

The exact SPK-1 pathology: a 90%-hot-key fact sort-merge-joined onto its dimension. Every hot-key row
lands in one reduce partition → **one straggler task** (`Duration` Max ≫ Median). AQE's skew-join
**splits** that fat partition at runtime — the *automatic* version of SPK-1's manual salting.

**Break it (AQE off):** `constrained` (AQE off, broadcast off) forces the sort-merge join.

In [5]:
N_COLD = 2_000
fact_b = skewed_keys(spark, n_rows=N_ROWS, hot_key_fraction=0.90,
                     n_cold_keys=N_COLD, hot_key=0, key_col="customer_id")
dim_b  = key_dimension(spark, n_keys=N_COLD + 1, key_col="customer_id")
join_b = fact_b.join(dim_b, on="customer_id", how="inner")

apply_profile(spark, "constrained")          # AQE off, broadcast off -> sort-merge join, one straggler
m_b_off = measure(spark, "skew: AQE off (SMJ)", lambda: join_b.count())
scan_plan(join_b, "(b) AQE OFF")
print(f"\nSKEW RATIO (max/median): {m_b_off['skew_ratio']}x   <- one straggler doing ~all the work")

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16
--- physical plan (b) AQE OFF ---
== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * Project (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Project (7)
            +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#232L]
Arguments: Range (0, 20000000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [3]: [id#232L AS row_id#233L, CASE WHEN (pmod(hash(id#232L, 42), 100) < 90) THEN 0 ELSE (pmod(id#232L, 2000) + 1) END AS customer_id#235L, (rand(23) * 100.0) AS amount#237]
Input [1]: [id#232L]

(3) Filter [codegen 

**Detect** (Spark UI → SQL/DataFrame → reduce Stage → Tasks → Summary Metrics): **Max ≫ Median**
task time, one fat **Shuffle Read** task. **Fix it (AQE on):** turn on skew-join and — because our
data is tiny — **lower the skew threshold to 16 MB** (the 256 MB default never trips on a laptop; see
[`SPK-1` §7](../skew/README.md)). Broadcast stays **off** so it remains a sort-merge join; the plan
gains an **`AQEShuffleRead ... skewed`** node and the skew ratio collapses.

In [6]:
apply_profile(spark, "tuned", **{
    "spark.sql.autoBroadcastJoinThreshold": "-1",                                  # keep it a sort-merge join
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "16m",          # laptop scale (SPK-1 trick)
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "2",
})
m_b_on = measure(spark, "skew: AQE skew-join", lambda: join_b.count())
scan_plan(join_b, "(b) AQE ON")
print("\n>>> Prove (b): skew ratio should collapse from tens-of-x toward ~1-3x")
compare([m_b_off, m_b_on])

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 200
  spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 16m
  spark.sql.adaptive.skewJoin.skewedPartitionFactor = 2
--- physical plan (b) AQE ON ---
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Filter (3)
      :        +- Project (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Range (6)


(1) Range
Output [1]: [id#232L]
Arguments: Range (0, 20000000, step=1, splits=Some(8))

(2) Project
Output [3]: [id#232L AS row_id#233L, CASE WHEN (pmod(hash(id#232L, 42), 100) < 90) THEN 0 ELSE (pmod(id

## Demo (c) — Runtime re-optimization: SMJ → broadcast

A big fact joined to a *small*, broadcastable dimension — but with broadcast **disabled** at plan
time, so Catalyst commits to a **sort-merge join** that shuffles *both* sides. Once the build side
materializes and AQE sees it's tiny, AQE **re-optimizes the join to a broadcast** — the big side's
`Exchange` vanishes.

**Break it (AQE off):** `constrained` (AQE off, broadcast off) → `SortMergeJoin` with two `Exchange` nodes.

In [7]:
fact_c = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_COLD, key_col="customer_id")
dim_c  = key_dimension(spark, n_keys=N_COLD, key_col="customer_id")   # small -> broadcastable
join_c = fact_c.join(dim_c, on="customer_id", how="inner")

apply_profile(spark, "constrained")          # AQE off, broadcast off -> SMJ, both sides shuffled
m_c_off = measure(spark, "reopt: AQE off (SMJ)", lambda: join_c.count())
scan_plan(join_c, "(c) AQE OFF")

# Fix it (AQE on): broadcast re-enabled (default 10 MB). AQE sees the small side at runtime and
# flips SortMergeJoin -> BroadcastHashJoin; the big side's Exchange disappears (shuffle ~ 0).
apply_profile(spark, "tuned")
m_c_on = measure(spark, "reopt: AQE on (broadcast)", lambda: join_c.count())
scan_plan(join_c, "(c) AQE ON")
print("\n>>> Prove (c): shuffle bytes should drop to ~0 as SMJ -> broadcast")
compare([m_c_off, m_c_on])

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16
--- physical plan (c) AQE OFF ---
== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * Project (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Project (7)
            +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#268L]
Arguments: Range (0, 20000000, step=1, splits=Some(8))

(2) Project [codegen id : 1]
Output [3]: [id#268L AS row_id#269L, pmod(id#268L, 2000) AS customer_id#271L, (rand(17) * 100.0) AS amount#273]
Input [1]: [id#268L]

(3) Filter [codegen id : 1]
Input [3]: [row_id#269L, customer_id#271L, amount#273]
Condi

## When AQE costs you — overhead & non-determinism

AQE is rarely *wrong*, but it isn't free:

1. **Planning overhead on tiny queries.** Re-planning after every shuffle is negligible on a big job
   but is pure overhead on a trivial one. Below we time a tiny aggregation **AQE off vs on** — on a
   query this small, AQE-on can be *no faster or slightly slower*. (Spark skips AQE entirely only for
   queries with **no** `Exchange`/subquery.)
2. **Run-to-run non-determinism.** The **coalesced partition count depends on the runtime data
   size**, so it can vary run-to-run and across environments — a trap if a downstream step, test, or
   file-count assertion expects a fixed number of partitions (ties to the streaming small-files
   concern in `STR-3`).

In [8]:
# A tiny query where AQE's re-planning is overhead, not benefit.
tiny = (uniform_keys(spark, n_rows=200_000, n_keys=50, key_col="k")
        .groupBy("k").agg(F.sum("amount").alias("total")))

apply_profile(spark, "constrained")                       # AQE off
m_tiny_off = measure(spark, "tiny: AQE off", lambda: tiny.count())

apply_profile(spark, "tuned")                             # AQE on (re-plans after the shuffle)
m_tiny_on = measure(spark, "tiny: AQE on", lambda: tiny.count())

print(">>> On a trivial query, AQE-on is no faster (and the partition count can vary run-to-run):")
compare([m_tiny_off, m_tiny_on])

Applied 'constrained' session profile:
  spark.sql.adaptive.enabled                       = false
  spark.sql.adaptive.skewJoin.enabled              = false
  spark.sql.adaptive.coalescePartitions.enabled    = false
  spark.sql.autoBroadcastJoinThreshold             = -1
  spark.sql.shuffle.partitions                     = 16
Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
>>> On a trivial query, AQE-on is no faster (and the partition count can vary run-to-run):
| Metric              | tiny: AQE off | tiny: AQE on |
| ------------------- | ------------- | ------------ |
| Wall-clock runtime  |        0.11 s |       0.08 s |
| Tasks               |            25 |           10 |
| Shuffle read        |       10.5 

## Takeaways & "in real production…"

- **AQE is on by default in Spark 3.2+ / 4.x.** Three fixes for free: **coalesce** (too many tiny
  partitions), **skew-join split** (one straggler key), and **SMJ → broadcast re-optimization** (a
  mis-sized join strategy).
- **Read it in the plan:** AQE-on plans say **`AdaptiveSparkPlan isFinalPlan=true`** and carry
  **`AQEShuffleRead`** nodes (annotated `coalesced` / `skewed`); the displayed plan is the *final*
  one. The **SQL / DataFrame** tab confirms the join operator and coalesced partition count
  (see [`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)).
- **Where to still intervene:** AQE adds a little planning overhead on trivial queries, and its
  coalesced partition counts are **non-deterministic** — don't hard-code an expected partition/file
  count downstream. On small data, lower `skewJoin.skewedPartitionThresholdInBytes` (the SPK-1 caveat).
- **In production:** keep AQE enabled; set `spark.sql.shuffle.partitions` sanely and let coalesce trim
  it; lean on AQE skew-join before hand-salting (`SPK-1`); set `autoBroadcastJoinThreshold`
  deliberately so the SMJ→broadcast re-optimization can fire.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We just restore
the production-tuned safety nets.

In [9]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")

Applied 'tuned' session profile:
  spark.sql.adaptive.enabled                       = true
  spark.sql.adaptive.skewJoin.enabled              = true
  spark.sql.adaptive.coalescePartitions.enabled    = true
  spark.sql.autoBroadcastJoinThreshold             = 10485760
  spark.sql.shuffle.partitions                     = 200
Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.
